# Práctica 02 — Medidas WiFi sobre planos

Notebook reproducible para analizar las capturas JSONL de la herramienta. No realiza nuevos escaneos. Antes de interpretar distancias, completa `coordinate_reference` y `access_points` en `config.json`, registra las coordenadas de las capturas con `--x` y `--y`, y conserva el plano local indicado en la configuración. Las capturas históricas sin posición se mantienen, pero no se usan para inventar distancias ni ubicaciones.

La señal en dBm y la calidad porcentual son métricas distintas: el notebook selecciona una sola métrica para cada AP y etiqueta sus unidades. Los gráficos solo muestran posiciones observadas; no interpolan cobertura.

In [ ]:
from collections import Counter
from html import escape
from pathlib import Path
import json
import sys

from IPython.display import HTML, display
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "wifi_scanner.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder or one of its subfolders.")

sys.path.insert(0, str(PROJECT_ROOT / "validation"))
from practica2_analysis import (
    build_observations,
    load_captures,
    observations_with_distances,
    plot_floorplan,
    plot_signal_distance,
    select_signal_metric,
    summarize_access_points,
    summarize_signal_methods,
)

local_config = PROJECT_ROOT / "config.json"
config_path = local_config if local_config.is_file() else PROJECT_ROOT / "config.example.json"
with config_path.open("r", encoding="utf-8") as config_file:
    config = json.load(config_file)
coordinate_reference = config.get("coordinate_reference", {})
access_points = config.get("access_points", [])
TARGET_PLAN_ID = None  # Set this when captures from several plans are present.
records = load_captures(PROJECT_ROOT / "data" / "practica2")
plan_counts = Counter(
    record.get("metadata", {}).get("position", {}).get("plan_id")
    for record in records
    if record.get("metadata", {}).get("position", {}).get("plan_id")
)
if TARGET_PLAN_ID is None and plan_counts:
    TARGET_PLAN_ID = sorted(plan_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
analysis_records = [
    record for record in records
    if TARGET_PLAN_ID is None
    or record.get("metadata", {}).get("position", {}).get("plan_id") == TARGET_PLAN_ID
]
observations = build_observations(analysis_records)

print(f"Configuration: {config_path.name}")
print(f"Capture records in selected plan: {len(analysis_records)} | AP observations: {len(observations)}")
print(f"Recorded sessions: {len({row['session_id'] for row in observations if row['session_id']})}")
print(f"Observations without plan coordinates: {sum(row['x'] is None or row['y'] is None for row in observations)}")
print(f"Selected plan: {TARGET_PLAN_ID or 'legacy records without a plan identifier'}")
print(f"Coordinate units: {coordinate_reference.get('units', 'units not set')}")

## Punto/zona y AP seleccionado

Si se dejan los selectores a `None`, se eligen el punto con más capturas y, dentro de él, el AP con más detecciones. Para repetir exactamente el análisis, fija aquí `TARGET_POINT_ID` y `TARGET_BSSID`. La tabla compara media en dBm, mediana en dBm y mediana tras convertir a mW y volver a dBm; para señal porcentual no se aplican esas conversiones.

In [ ]:
TARGET_POINT_ID = None
TARGET_BSSID = None

capture_counts = Counter(
    (record.get("metadata", {}).get("position", {}).get("point_id")
     or record.get("metadata", {}).get("room"))
    for record in analysis_records
    if (record.get("metadata", {}).get("position", {}).get("point_id")
        or record.get("metadata", {}).get("room"))
)
if TARGET_POINT_ID is None and capture_counts:
    TARGET_POINT_ID = sorted(capture_counts.items(), key=lambda item: (-item[1], str(item[0])))[0][0]
if TARGET_POINT_ID is None:
    raise ValueError("No measurement point is available in the capture data.")

point_summary = summarize_access_points(observations, TARGET_POINT_ID)
configured_bssids = {ap.get("bssid") or ap.get("bssid_hash") for ap in access_points}
if TARGET_BSSID is None:
    TARGET_BSSID = next(
        (row["bssid"] for row in point_summary if row["bssid"] in configured_bssids),
        point_summary[0]["bssid"] if point_summary else None,
    )
if TARGET_BSSID is None:
    raise ValueError(f"No AP observations were found at point '{TARGET_POINT_ID}'.")

print(f"Selected point: {TARGET_POINT_ID}")
print(f"Selected AP BSSID/MAC: {TARGET_BSSID}")
print(f"Capture records at selected point: {capture_counts[TARGET_POINT_ID]}")

def show_table(rows):
    if not rows:
        display(HTML("<p>No rows to display.</p>"))
        return
    columns = list(rows[0])
    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

show_table([
    {"point_id": point_id, "capture_records": count}
    for point_id, count in sorted(capture_counts.items(), key=lambda item: str(item[0]))
])
show_table(point_summary)

signal_methods = summarize_signal_methods(observations, TARGET_POINT_ID, TARGET_BSSID)
show_table([signal_methods])
if signal_methods["valid_rssi_observations"] == 0:
    print("This AP has no RSSI in dBm; percentage quality is reported separately and cannot be converted to dBm.")

## Señal observada frente a distancia

La distancia euclídea se calcula entre las coordenadas de cada captura y las coordenadas del AP, solo cuando pertenecen al mismo plano. Si `meters_per_unit` no está calibrado, la distancia se expresa en las unidades del plano (por ejemplo, píxeles), no en metros.

In [ ]:
metric, metric_units = select_signal_metric(observations, TARGET_BSSID)
if metric is None:
    raise ValueError(f"AP '{TARGET_BSSID}' has no usable signal values.")

distance_observations = observations_with_distances(
    observations,
    access_points,
    TARGET_BSSID,
    default_plan_id=coordinate_reference.get("plan_id"),
    default_meters_per_unit=coordinate_reference.get("meters_per_unit"),
)
plot_rows = [row for row in distance_observations if row.get(metric) is not None]
print(f"Signal metric: {metric} ({metric_units})")
print(f"Observations with signal and valid AP distance: {len(plot_rows)}")
if plot_rows:
    figure = plot_signal_distance(distance_observations, TARGET_BSSID, metric, metric_units)
    display(figure)
    plt.close(figure)
else:
    print("Distance chart unavailable: configure measured coordinates and this AP's position in config.json.")

## Posiciones observadas sobre el plano

El color representa la mediana de las observaciones válidas del AP seleccionado en cada coordenada. No se asignan valores a zonas sin medidas. Verifica que el plano y la referencia de coordenadas correspondan al mismo edificio y planta.

In [ ]:
map_name = coordinate_reference.get("map_file")
map_path = (PROJECT_ROOT / map_name) if map_name else None
if not plot_rows:
    print("Plan chart unavailable until AP and measurement coordinates are configured.")
elif map_path is None or not map_path.is_file():
    print(f"Plan chart unavailable: configure map_file and keep the image locally (configured: {map_name!r}).")
else:
    figure = plot_floorplan(
        observations,
        access_points,
        TARGET_BSSID,
        metric,
        metric_units,
        map_path,
        default_plan_id=coordinate_reference.get("plan_id"),
    )
    display(figure)
    plt.close(figure)

## Contraste y calidad de datos

Usa las filas siguientes para contrastar manualmente al menos dos o tres observaciones con sus registros JSONL originales (BSSID/MAC, instante, posición y señal). Una posición ausente o una señal no disponible se conserva como ausencia; no se sustituye por cero. Compara también los recuentos del resumen con las detecciones y conexiones.

In [ ]:
validation_rows = [
    {
        "timestamp_utc": row["timestamp_utc"],
        "session_id": row["session_id"],
        "sample_index": row["sample_index"],
        "point_id": row["point_id"],
        "plan_id": row["plan_id"],
        "x": row["x"],
        "y": row["y"],
        "measurement_procedure": row["measurement_procedure"],
        "measurement_incidents": row["measurement_incidents"],
        "bssid": row["bssid"],
        "connected": row["connected"],
        metric: row[metric],
        "distance": row.get("distance"),
    }
    for row in sorted(
        (item for item in plot_rows if item["bssid"] == TARGET_BSSID),
        key=lambda item: (item["timestamp_utc"] or "", item["sample_index"] or 0),
    )[:3]
]
show_table(validation_rows)

connected_count = sum(row["connected"] for row in observations if row["point_id"] == TARGET_POINT_ID)
missing_signal_count = sum(
    row[metric] is None for row in observations
    if row["point_id"] == TARGET_POINT_ID and row["bssid"] == TARGET_BSSID
)
print(f"Connected captures at selected point: {connected_count}")
print(f"Selected AP observations missing {metric}: {missing_signal_count}")
print("Check these rows against the original JSONL and record any exclusions/incidents before submitting.")